# Task 1 — class imbalance check (v4)

The slide suggests class weighting or resampling. Test one direct class-weight intervention on the established 38-feature LightGBM using the same 3-fold × 2-repeat outer folds. The fixed 0.40 threshold isolates the effect of class weights; a gain would require a subsequent matched inner-threshold check before selection. No Test fitting or leaderboard use.

In [1]:
from pathlib import Path
import sys,json,joblib
ROOT=next(p for p in (Path.cwd(),*Path.cwd().parents) if (p/'src'/'pipeline'/'save_model.py').exists())
if str(ROOT) not in sys.path:sys.path.insert(0,str(ROOT))
from src.pipeline.random_seed import set_seed
from src.pipeline.save_model import save_model
from src.pipeline.evaluate_model import evaluate_model
SEED=set_seed()
import numpy as np,pandas as pd
from sklearn.base import clone
from sklearn.model_selection import RepeatedStratifiedKFold,FixedThresholdClassifier
from sklearn.metrics import fbeta_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
SRC=ROOT/'src';TASK=SRC/'task1';RES=TASK/'models'/'results';SAVE=TASK/'save'
DATA='eda_error_context_v2';NAME='t1_lgbm_balanced_fixed040_cv6_v4'
card=json.loads((TASK/'data'/'eda'/f'{DATA}.json').read_text(encoding='utf-8'))
train=pd.read_parquet(SRC/card['splits']['train']['path'])
selected=json.loads((SAVE/'t1_lgbm_combined_pruned_cv6_full_v1.json').read_text(encoding='utf-8'))
features=list(selected['card']['selected_features'])
source_card=json.loads((SAVE/'t1_lgbm_original_tuned_v1.json').read_text(encoding='utf-8'))
source=joblib.load(SRC/source_card['model_path'])
y=train.is_cheater.astype(int);X=train[features]
splits=list(RepeatedStratifiedKFold(n_splits=3,n_repeats=2,random_state=SEED).split(X,y))
assert len(features)==38 and len(splits)==6
print(len(train),'rows',len(features),'features')

90816 rows 38 features


## 1. Reconstruct fixed-threshold baseline

In [2]:
parts=[]
for stem in ['round4_lgbm_combined_pruned_oof_v1','round4_lgbm_combined_pruned_repeat2_oof_v1']:
    part=pd.read_parquet(RES/f'{stem}.parquet');parts.append(part)
baseline=pd.concat(parts,ignore_index=True)
assert len(baseline)==2*len(train)
def fold_score(frame):
    return np.array([fbeta_score(g.true,g.pred,beta=2,average='macro',zero_division=0)
                     for _,g in frame.groupby('fold',sort=True)])
baseline['pred']=(baseline.prob>=.4).astype(int)
base=fold_score(baseline)
print('Fixed-threshold baseline:',base.round(6),base.mean())

Fixed-threshold baseline: [0.778309 0.777654 0.777587 0.77484  0.77636  0.777554] 0.7770508194890211


## 2. Fit class-weighted LightGBM, save, and evaluate

`class_weight='balanced'` is computed inside each training fold by LightGBM. All full-data preprocessing and the fixed threshold are saved with the model. The six-fold score is a procedure estimate; both repeats reuse the same accounts.

In [3]:
paths=[SAVE/f'{NAME}.joblib',SAVE/f'{NAME}.json',SAVE/f'{NAME}-evaluation.json']
cache=RES/f'{NAME}-oof.parquet';meta=cache.with_suffix('.json')
key={'dataset':DATA,'dataset_created_at':card['created_at'],'source_created_at':source_card['created_at'],
     'features':features,'seed':SEED,'threshold':.4,'class_weight':'balanced'}
if cache.exists()!=meta.exists():raise RuntimeError('Incomplete OOF cache')
if cache.exists():
    assert json.loads(meta.read_text(encoding='utf-8'))==key
    oof=pd.read_parquet(cache)
else:
    if any(p.exists() for p in paths):raise RuntimeError('Saved model without OOF cache')
    rows=[]
    for i,(fit,check) in enumerate(splits,1):
        m=clone(source).set_params(class_weight='balanced')
        m.fit(X.iloc[fit],y.iloc[fit])
        prob=m.predict_proba(X.iloc[check])[:,1]
        rows.append(pd.DataFrame({'fold':i,'row_index':check,'true':y.iloc[check].to_numpy(),
                                  'prob':prob,'pred':(prob>=.4).astype(int)}))
        print('Fold',i,'fitted',flush=True)
    oof=pd.concat(rows,ignore_index=True)
    cache.parent.mkdir(parents=True,exist_ok=True);oof.to_parquet(cache,index=False)
    meta.write_text(json.dumps(key,indent=2),encoding='utf-8')
for i,(_,check) in enumerate(splits,1):
    part=oof[oof.fold==i].sort_values('row_index')
    assert np.array_equal(part.row_index,np.sort(check))
    assert np.array_equal(part.true,y.iloc[part.row_index])
weighted=fold_score(oof)
if not any(p.exists() for p in paths):
    selector=ColumnTransformer([('keep','passthrough',features)],remainder='drop',
                               verbose_feature_names_out=False).set_output(transform='pandas')
    model=Pipeline([('select',selector),('classifier',FixedThresholdClassifier(
        estimator=clone(source).set_params(class_weight='balanced'),threshold=.4))])
    model.fit(train.drop(columns=['player_id','is_cheater']),y)
    save_model(task=1,name=NAME,model=model,data_name=DATA,
        card={'idea':'Balanced LightGBM on selected 38 features','seed':SEED,'features':features,
              'threshold':.4,'validation':'3-fold x 2 repeats, fixed threshold'})
    evaluate_model(task=1,name=NAME,score=float(weighted.mean()),
        metric_name='macro_F2_outer_cv6_fixed040',higher_is_better=True,
        card={'fold_scores':weighted.tolist(),'same_fold_baseline':base.tolist(),
              'note':'Procedure estimate, not independent full-model holdout score'})
else:
    assert all(p.exists() for p in paths)
    result=json.loads(paths[2].read_text(encoding='utf-8'))
    assert abs(result['score']-weighted.mean())<1e-12
display(pd.DataFrame({'baseline':base,'balanced':weighted,'delta':weighted-base},index=range(1,7)).round(6))
print('Mean delta:',(weighted-base).mean(),'wins:',int((weighted>base).sum()),'/6')

,baseline,balanced,delta
1,0.778309,0.759895,-0.018414
2,0.777654,0.757061,-0.020593
3,0.777587,0.761605,-0.015982
4,0.774840,0.755352,-0.019488
5,0.776360,0.762010,-0.014350
6,0.777554,0.762524,-0.015030


Mean delta: -0.017309666054463175 wins: 0 /6


## 3. Threshold headroom diagnostic

Balanced class weights shift the probability scale, so 0.40 is not a fair final threshold. The following same-OOF sweep is **optimistic** because each threshold is scored on the rows used to select it. It is only a cheap screen for whether a costly nested threshold experiment has meaningful headroom; it is not a validation score for a new model.

In [4]:
threshold_rows=[]
for threshold in np.arange(.40,.601,.01):
    probe=oof.copy();probe['pred']=(probe.prob>=threshold).astype(int)
    fold_values=fold_score(probe)
    threshold_rows.append({'threshold':float(threshold),'mean_macro_F2':float(fold_values.mean()),
                           'fold_wins_vs_fixed_baseline':int((fold_values>base).sum())})
headroom=pd.DataFrame(threshold_rows).sort_values('mean_macro_F2',ascending=False)
display(headroom.head(5).round(6))
print('Selected model threshold-tuned outer CV: 0.77733; optimistic weighted best:',
      round(headroom.iloc[0].mean_macro_F2,6))

,threshold,mean_macro_F2,fold_wins_vs_fixed_baseline
13,0.53,0.777652,4
11,0.51,0.777369,4
12,0.52,0.777364,4
14,0.54,0.777233,3
10,0.50,0.776829,4


Selected model threshold-tuned outer CV: 0.77733; optimistic weighted best: 0.777652


## Decision

If weighting wins only marginally or inconsistently at the fixed threshold, keep the selected Task 1 model. A clear gain requires a matched inner-threshold comparison before using this candidate for submission.